# 🎯 실습: RunnableParallel과 체인 결합


 `chain = prompt | llm | parser` → 한 가지 결과 (분석만)   
 `RunnableParallel(분석, 답장, 분류, ...)` → 여러 결과를 한 번에

같은 입력으로 **여러 작업을 동시에** 처리하는 법을 배웁니다.

## 📋 빌드업 흐름

1교시 체인을 출발점으로 부품이 하나씩 추가됩니다.

```
[1교시] 분석 체인
   chain_analysis = prompt | llm | PydanticOutputParser

        ↓ Step 2 답장 체인 추가
[추가 ①] StrOutputParser로 답장 체인
   chain_reply = prompt | llm | StrOutputParser()

        ↓ Step 3 ⭐ 두 체인을 묶기
[추가 ②] RunnableParallel
   full = RunnableParallel(analysis=..., reply=...)

        ↓ Step 4 일반 함수도 끼워넣기
[추가 ③] RunnableLambda
   chain | RunnableLambda(my_function)

        ↓ Step 5 원본 데이터 보존
[추가 ④] RunnablePassthrough
   RunnableParallel(result=chain, original=RunnablePassthrough())

        ↓ Step 6 ⭐ 부품을 모두 결합해 점점 더 풍성한 파이프라인으로
[응용] 리뷰 응대 파이프라인 v1 → v2 → v3 → v4
   각 단계마다 이전 파이프라인을 확장
```

## 📋 노트북 순서

```
Step 0. 환경 설정
Step 1. 1교시 체인 복습              ← 분석 체인 다시 만들기
Step 2. ① 답장 체인 추가             ← StrOutputParser 활용
Step 3. ② RunnableParallel ⭐       ← 두 체인을 묶기
Step 4. ③ RunnableLambda             ← 후처리 함수 끼우기
Step 5. ④ RunnablePassthrough        ← 원본 데이터 보존
Step 6. ⭐ 리뷰 응대 파이프라인 점진 완성 (v1→v4)
Step 7. 도전 과제
```

준비됐으면 시작!


---

# Step 0. 환경 설정


In [ ]:
#!uv add langchain langchain-openai python-dotenv pydantic

## 공통 import

1교시에서 쓴 도구 + 오늘 새로 배울 도구.


In [1]:
from typing import Literal, Optional
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, StrOutputParser

# 오늘 새로 배우는 도구 3가지
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ import 완료")

✅ import 완료


---

# Step 1. 1교시 체인 복습 — 분석 체인



In [2]:
# 1교시 스키마 (간단 버전)
class ReviewAnalysis(BaseModel):
    rating: int = Field(description="평점 1-5점", ge=1, le=5)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감정")
    keywords: list[str] = Field(description="핵심 키워드 3개")

# 1교시 5단계 패턴
parser_a = PydanticOutputParser(pydantic_object=ReviewAnalysis)

prompt_a = ChatPromptTemplate.from_messages([
    ("system", "리뷰 분석가입니다. 한국어로 답하세요.\n\n{format_instructions}"),
    ("human", "리뷰: {review}")
]).partial(format_instructions=parser_a.get_format_instructions())

# 1교시 체인 (분석)
chain_analysis = prompt_a | llm | parser_a

print("✅ 분석 체인 준비 완료 (1교시 복습)")

✅ 분석 체인 준비 완료 (1교시 복습)


In [3]:
# 동작 확인
result = chain_analysis.invoke({"review": "이 카메라 정말 좋아요! 가격도 싸고 화질도 최고!"})
print(f"⭐ 평점: {result.rating}/5")
print(f"😊 감정: {result.sentiment}")
print(f"🏷️  키워드: {', '.join(result.keywords)}")

⭐ 평점: 5/5
😊 감정: positive
🏷️  키워드: 카메라, 가격, 화질


**✅ 분석 체인 동작 확인 완료**

이제 이 체인을 **출발점**으로, 오늘 새 부품을 하나씩 추가해봅시다.


---

# Step 2. ① 답장 체인 추가 — `StrOutputParser` 활용

쇼핑몰 운영자 입장에서 리뷰를 받으면 분석만 하면 끝이 아니죠. **답장도 자동으로** 보내고 싶습니다.

답장은 자연어 문자열로 받으면 충분하니까 1교시에서 살짝 본 `StrOutputParser` 를 씁니다.

> 💡 **OutputParser 선택의 원칙**: 정형 데이터가 필요하면 `PydanticOutputParser`, 자연어 텍스트 그대로 쓸 거면 `StrOutputParser`. 오늘 만드는 모든 체인이 둘 중 하나로 끝납니다.

## 2-1. 답장 체인 만들기


In [3]:
# 답장 프롬프트
prompt_r = ChatPromptTemplate.from_messages([
    ("system",
     "쇼핑몰 고객센터 응대 담당자입니다. "
     "리뷰에 진심을 담아 한국어로 3-4문장의 답장을 작성하세요. "
     "긍정 리뷰엔 감사를, 부정 리뷰엔 사과와 개선 의지를 표현하세요."),
    ("human", "고객 리뷰: {review}")
])

# 답장 체인
chain_reply = prompt_r | llm | StrOutputParser()
#                                ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑ 1교시와 다른 부품 (자연어용)
print("✅ 답장 체인 준비 완료")

✅ 답장 체인 준비 완료


In [4]:
# 동작 확인
reply = chain_reply.invoke({"review": "이 카메라 정말 좋아요!"})
print(reply)

고객님, 소중한 리뷰 감사합니다! 카메라가 마음에 드신다니 정말 기쁩니다. 앞으로도 더 좋은 제품과 서비스로 보답할 수 있도록 노력하겠습니다. 언제든지 궁금한 점이 있으시면 문의해 주세요!


**✅ 두 체인이 모두 준비됐습니다**

| 체인 | OutputParser | 출력 |
| --- | --- | --- |
| `chain_analysis` | `PydanticOutputParser` | `ReviewAnalysis` 객체 |
| `chain_reply` | `StrOutputParser` | `str` (자연어 답장) |

이제 두 체인을 합칠 차례예요.


---

# Step 3. ② `RunnableParallel` — 두 체인을 묶기 ⭐

## 3-1. 단순한 방법 vs RunnableParallel

먼저 두 체인을 **단순한 방법**으로 호출해봅시다.


In [5]:
import time
review = "이 카메라 정말 좋아요!"

# 단순한 방법: 순차 호출
start = time.time()
analysis = chain_analysis.invoke({"review": review})
reply = chain_reply.invoke({"review": review})
sequential_time = time.time() - start

print(f"⏱️  순차 호출: {sequential_time:.2f}초")
print(f"   분석: {analysis.sentiment}, 평점 {analysis.rating}")
print(f"   답장: {reply[:50]}...")

⏱️  순차 호출: 3.43초
   분석: positive, 평점 5
   답장: 고객님, 소중한 리뷰 감사합니다! 카메라가 마음에 드신다니 정말 기쁩니다. 앞으로도 더 좋...


**문제점**

- 분석과 답장이 **차례로** 실행됨 → 시간 = 분석 시간 + 답장 시간
- 결과를 따로따로 받아서 합치는 코드가 필요

이걸 LCEL 방식으로 깔끔하게 만드는 게 `RunnableParallel`입니다.

## 3-2. RunnableParallel로 한 번에


In [6]:
# 두 체인을 병렬로 묶기
full_chain = RunnableParallel(
    analysis=chain_analysis,
    reply=chain_reply
)

# 한 번 호출
start = time.time()
result = full_chain.invoke({"review": review})
parallel_time = time.time() - start

print(f"⏱️  병렬 호출: {parallel_time:.2f}초")
print(f"📊 속도 향상: {sequential_time / parallel_time:.1f}배")
print()
print(f"📦 result 타입: {type(result).__name__}")
print(f"   .keys(): {list(result.keys())}")

⏱️  병렬 호출: 1.28초
📊 속도 향상: 2.7배

📦 result 타입: dict
   .keys(): ['analysis', 'reply']


**👀 변화 확인**

- 두 작업이 **동시에** 실행 → 보통 1.5~2배 빠름
- 결과가 **dict** 로 깔끔하게 묶여 나옴

체인이 많을수록 차이가 커집니다.

| 체인 수 | 순차 (예상) | 병렬 (예상) |
| --- | --- | --- |
| 2개 | 4초 | 2초 |
| 5개 | 10초 | 2-3초 |

## 3-3. 결과 활용

dict로 묶였으니 키로 꺼냅니다.


In [7]:
# RunnableParallel 결과는 dict
print(f"⭐ 평점: {result['analysis'].rating}/5")
print(f"😊 감정: {result['analysis'].sentiment}")
print(f"🏷️  키워드: {', '.join(result['analysis'].keywords)}")
print()
print(f"💬 자동 답장:")
print(f"   {result['reply']}")

⭐ 평점: 5/5
😊 감정: positive
🏷️  키워드: 카메라, 좋아요, 추천

💬 자동 답장:
   고객님, 소중한 리뷰 감사합니다! 카메라가 마음에 드신다니 정말 기쁩니다. 앞으로도 더 좋은 제품과 서비스로 보답할 수 있도록 노력하겠습니다. 언제든지 궁금한 점이 있으시면 문의해 주세요!


**핵심 패턴 정리**

```python
# RunnableParallel — 같은 입력을 여러 체인에 동시 적용
full = RunnableParallel(
    분석=chain_analysis,
    답장=chain_reply,
    카테고리=chain_category,    # 얼마든지 추가 가능
)
```

이게 오늘의 가장 중요한 패턴입니다. **Step 6에서 이 패턴을 점점 확장해서 풍성한 파이프라인을 만들 거예요.**


---

# Step 4. ③ `RunnableLambda` — 일반 함수도 체인에

LLM 호출이 아닌 **일반 Python 함수**를 체인에 끼울 수 있어요.

## 4-1. 사용 예: 평점에 따라 경고 표시

분석 결과의 평점이 낮으면 자동으로 경고를 붙여주는 함수를 만들어봅시다.


In [8]:
# 일반 Python 함수 — 분석 결과를 받아 경고 마크 부착
def add_alert(analysis):
    if analysis.rating <= 2:
        return {"level": "🚨 긴급 응대", "data": analysis}
    elif analysis.rating <= 3:
        return {"level": "⚠️ 주의 필요", "data": analysis}
    else:
        return {"level": "🟢 정상", "data": analysis}

# RunnableLambda로 감싸기
alert_step = RunnableLambda(add_alert)

# 체인 뒤에 끼워넣기
chain_with_alert = chain_analysis | alert_step

# 테스트
for review in ["완전 만족!", "그냥 그래요", "최악이에요. 환불해주세요!"]:
    r = chain_with_alert.invoke({"review": review})
    print(f"{r['level']:15s} | {review}")

🟢 정상            | 완전 만족!
⚠️ 주의 필요        | 그냥 그래요
🚨 긴급 응대         | 최악이에요. 환불해주세요!


**👀 관찰 포인트**

- `chain_analysis` (1교시 분석 체인) 뒤에 일반 함수가 자연스럽게 이어짐
- LLM 호출과 일반 로직이 **하나의 흐름**으로 묶임
- Step 6-3에서 이 패턴을 본격 활용합니다.

> 💡 `chain | my_function` 도 자동으로 RunnableLambda로 감싸져 동작하지만, **명시적으로 쓰는 것이 의도가 명확**해서 권장합니다.


---

# Step 5. ④ `RunnablePassthrough` — 원본 데이터 보존

`RunnableParallel` 결과에 **원본 입력**을 함께 두고 싶을 때가 있어요.


In [9]:
# 분석 결과 + 원본 리뷰를 모두 한 번에
full = RunnableParallel(
    analysis=chain_analysis,
    reply=chain_reply,
    original=RunnablePassthrough()    # 입력을 그대로 통과시킴
)

result = full.invoke({"review": "포장이 너무 아쉬워요."})
print(f"📝 원본 입력: {result['original']}")
print(f"⭐ 평점: {result['analysis'].rating}")
print(f"💬 답장: {result['reply'][:60]}...")

📝 원본 입력: {'review': '포장이 너무 아쉬워요.'}
⭐ 평점: 3
💬 답장: 고객님, 소중한 의견 감사합니다. 포장에 대한 불편을 드려 정말 죄송합니다. 고객님의 소중한 피드백을 바탕으...


**👀 관찰 포인트**

- `result["original"]` 에 **입력 그대로** `{"review": "..."}` 가 보존됨
- 분석 결과, 답장, 원본을 모두 한 번에 받을 수 있음

이게 왜 유용할까요? **로그 저장, 데이터베이스 기록** 등에서 "원본 + 결과"를 함께 저장해야 할 때가 많아요. Step 6-4에서 이 패턴이 본격 활용됩니다.

**핵심 패턴 정리**

| 도구 | 역할 |
| --- | --- |
| `RunnablePassthrough()` | 입력을 그대로 통과 (변환 없음) |
| `RunnableParallel(...)` | 입력을 여러 체인에 동시 전달 |
| `RunnableLambda(함수)` | 일반 함수를 체인에 끼워넣기 |

이 4가지 (Parser 2개 + Runnable 3개)가 오늘의 부품 전부입니다. **이제 모두 모아 진짜 파이프라인을 만들어봅시다.**


---

# Step 6. ⭐ 리뷰 응대 파이프라인 점진 완성

오늘 배운 부품들을 모두 결합해 **진짜 쓸 수 있는 리뷰 응대 시스템**을 점진적으로 완성합니다.

```
v1: 분석 + 답장 + 카테고리         (Step 3 RunnableParallel 확장)
   ↓ 키워드 분석 체인 추가
v2: v1 + 키워드                    (4개 체인 동시)
   ↓ 알림 라우팅 로직 추가
v3: v2 결과 → 알림 분기            (Step 4 RunnableLambda + Pydantic 결과)
   ↓ 원본 입력도 함께 보존
v4: v3 + 원본 = 완전한 로그 레코드  (Step 5 RunnablePassthrough + LogRecord)
```

**각 버전이 이전 버전을 그대로 가져다 확장**합니다. 따로 만드는 게 아니에요.

> 💡 **OutputParser 일관 사용**: 모든 LLM 체인은 `PydanticOutputParser` 또는 `StrOutputParser`로 끝납니다. Python 함수(`RunnableLambda`) 결과도 Pydantic 클래스로 받아 **일관된 정형 출력**을 유지해요.


## 6-1. v1 — 분석 + 답장 + 카테고리 (RunnableParallel 3개)

먼저 카테고리 분류 체인을 새로 만듭니다. **1교시 5단계 패턴 그대로**.


In [10]:
# 카테고리 분류 스키마 (1교시 패턴)
class ReviewCategory(BaseModel):
    main_topic: Literal["품질", "가격", "배송", "디자인", "기타"] = Field(description="주요 토픽")
    priority: Literal["high", "medium", "low"] = Field(description="대응 우선순위")

parser_c = PydanticOutputParser(pydantic_object=ReviewCategory)
prompt_c = ChatPromptTemplate.from_messages([
    ("system", "리뷰를 분류하고 우선순위를 매깁니다. 한국어로.\n\n{format_instructions}"),
    ("human", "리뷰: {review}")
]).partial(format_instructions=parser_c.get_format_instructions())

chain_category = prompt_c | llm | parser_c
print("✅ 카테고리 분류 체인 준비 (chain_category)")

✅ 카테고리 분류 체인 준비 (chain_category)


In [11]:
# v1: Step 3 패턴을 그대로 확장 — 3개 체인 병렬
review_pipeline_v1 = RunnableParallel(
    analysis=chain_analysis,    # Step 1
    reply=chain_reply,          # Step 2
    category=chain_category,    # 방금 추가
)
print("✅ review_pipeline_v1 (3개 체인) 완성")

✅ review_pipeline_v1 (3개 체인) 완성


In [12]:
# 테스트 — 3건 리뷰
reviews = [
    "이 카메라 정말 좋아요! 가격도 만족스럽고요.",
    "배송이 너무 늦어요. 답답합니다.",
    "디자인은 예쁜데 사용법이 어려워요.",
]

for r in reviews:
    res = review_pipeline_v1.invoke({"review": r})
    icon = {"high": "🚨", "medium": "⚠️", "low": "📌"}[res["category"].priority]
    print(f"{icon} [{res['category'].main_topic}] ⭐{res['analysis'].rating}/5 ({res['analysis'].sentiment})")
    print(f"   📝 '{r}'")
    print(f"   💬 {res['reply'][:60]}...")
    print()

🚨 [가격] ⭐5/5 (positive)
   📝 '이 카메라 정말 좋아요! 가격도 만족스럽고요.'
   💬 고객님, 소중한 리뷰 감사합니다! 카메라에 만족하셨다니 저희도 매우 기쁩니다. 앞으로도 좋은 제품과 서비스를...

🚨 [배송] ⭐2/5 (negative)
   📝 '배송이 너무 늦어요. 답답합니다.'
   💬 고객님, 불편을 드려 정말 죄송합니다. 배송 지연으로 인해 답답한 마음을 느끼셨다니 안타깝습니다. 저희는 이...

⚠️ [디자인] ⭐3/5 (neutral)
   📝 '디자인은 예쁜데 사용법이 어려워요.'
   💬 고객님, 소중한 리뷰 감사합니다. 디자인이 마음에 드셨다니 기쁩니다. 사용법이 어려우셨다니 죄송합니다. 고객...



**👀 v1 관찰 포인트**

- **호출 한 번에 3가지 결과**: 분석 객체 + 답장 문자열 + 카테고리 객체
- **출력 타입이 달라도 OK**: Pydantic 객체, str, Pydantic 객체가 한 dict에 묶임
- 이게 RunnableParallel의 진짜 매력

다음 v2에서는 여기에 **키워드 분석 체인**을 하나 더 붙입니다.


## 6-2. v2 — v1에 키워드 체인 추가 (4개 동시)

쇼핑몰 운영자가 "어떤 단어가 자주 나오는지" 추적하려면 **키워드 추출**이 필요해요.

기존 `ReviewAnalysis.keywords` 와 별개로, **마케팅용 해시태그 5개**를 추출하는 체인을 추가합니다.


In [13]:
# 마케팅용 키워드 스키마 (1교시 5단계 패턴 그대로)
class MarketingKeywords(BaseModel):
    hashtags: list[str] = Field(description="SNS 마케팅용 해시태그 5개 (# 없이)")
    sentiment_score: int = Field(description="긍정도 0-100점", ge=0, le=100)

parser_m = PydanticOutputParser(pydantic_object=MarketingKeywords)
prompt_m = ChatPromptTemplate.from_messages([
    ("system", "리뷰에서 마케팅용 해시태그를 뽑고 긍정도를 점수화하세요.\n\n{format_instructions}"),
    ("human", "리뷰: {review}")
]).partial(format_instructions=parser_m.get_format_instructions())

chain_marketing = prompt_m | llm | parser_m
print("✅ 마케팅 키워드 체인 준비 (chain_marketing)")

✅ 마케팅 키워드 체인 준비 (chain_marketing)


In [14]:
# v2: v1에 chain_marketing 하나만 추가 — 4개 체인 동시
review_pipeline_v2 = RunnableParallel(
    analysis=chain_analysis,
    reply=chain_reply,
    category=chain_category,
    marketing=chain_marketing,    # ← v1 대비 추가된 부품
)
print("✅ review_pipeline_v2 (4개 체인) 완성")

✅ review_pipeline_v2 (4개 체인) 완성


In [18]:
# 테스트 — 긍정 리뷰
res = review_pipeline_v2.invoke({"review": "이 카메라 진짜 좋아요! 가성비 최고고 화질도 훌륭!"})

print(f"⭐ 평점:    {res['analysis'].rating}/5  |  📁 카테고리: {res['category'].main_topic}")
print(f"😊 감정:    {res['analysis'].sentiment}  |  📊 긍정도: {res['marketing'].sentiment_score}/100")
print(f"🏷️  분석 키워드:   {', '.join(res['analysis'].keywords)}")
print(f"📱 마케팅 해시태그: {', '.join('#' + h for h in res['marketing'].hashtags)}")
print(f"💬 답장: {res['reply'][:80]}...")

⭐ 평점:    5/5  |  📁 카테고리: 품질
😊 감정:    positive  |  📊 긍정도: 95/100
🏷️  분석 키워드:   가성비, 화질, 카메라
📱 마케팅 해시태그: #카메라, #가성비, #화질, #추천, #최고
💬 답장: 고객님, 소중한 리뷰 감사합니다! 카메라의 가성비와 화질에 만족하셨다니 정말 기쁩니다. 앞으로도 고객님께서 만족하실 수 있는 제품을 제공하기 위...


**👀 v2 관찰 포인트**

- v1에 **딱 한 줄(`marketing=chain_marketing`)만 추가**해서 기능 확장
- 같은 리뷰에서 4가지 정보가 동시 추출됨
- **모든 LLM 체인이 OutputParser로 끝나는 패턴**이 유지됨 (Pydantic 3개 + Str 1개)

다음 v3에서는 v2의 결과를 보고 **자동 알림 분기 처리**를 추가합니다.


## 6-3. v3 — v2 + 알림 라우팅 (RunnableLambda + Pydantic 결과)

v2가 4가지 결과를 만들었어요. 이 결과를 보고 **자동 알림**을 보낸다고 해봅시다.

- 부정 리뷰 + 저평점 → CS팀에 긴급 알림
- 긍정 리뷰 + 고평점 → 마케팅팀에 공유
- 그 외 → 일반 로그

이 분기 로직은 Python 함수가 자연스럽죠. Step 4에서 배운 `RunnableLambda` 등장 차례.

> 💡 **OutputParser 일관성 포인트**: 함수가 반환하는 값도 **Pydantic 클래스**로 받아 정형성을 유지합니다. dict보다 검증·자동완성·안정성이 좋아져요.


In [15]:
# 알림 액션 스키마 — 함수 결과도 Pydantic으로!
class AlertAction(BaseModel):
    action: str = Field(description="수행할 액션")
    channel: Literal["urgent", "marketing", "logs"] = Field(description="알림 채널")
    auto_reply: str = Field(description="고객에게 보낼 답장")
    summary: str = Field(description="내부용 한 줄 요약")

print("✅ AlertAction 스키마 준비")

✅ AlertAction 스키마 준비


In [16]:
# 분석 결과 → AlertAction 으로 변환하는 함수
def route_alert(parallel_result) -> AlertAction:
    analysis = parallel_result["analysis"]
    marketing = parallel_result["marketing"]
    category = parallel_result["category"]
    reply = parallel_result["reply"]

    if analysis.sentiment == "negative" and analysis.rating <= 2:
        return AlertAction(action="🚨 CS팀 즉시 호출", channel="urgent",
                           auto_reply=reply,
                           summary=f"[{category.main_topic}] {analysis.rating}/5점 불만 리뷰")
    elif marketing.sentiment_score >= 80:
        return AlertAction(action="🎉 마케팅팀에 공유", channel="marketing",
                           auto_reply=reply,
                           summary=f"긍정도 {marketing.sentiment_score}점 — 해시태그 활용 가능")
    else:
        return AlertAction(action="📋 일반 로그 저장", channel="logs",
                           auto_reply=reply,
                           summary=f"[{category.main_topic}] {analysis.rating}/5점")

In [17]:
# v3: v2 + RunnableLambda — v2를 그대로 가져다 함수만 이어붙임
review_pipeline_v3 = review_pipeline_v2 | RunnableLambda(route_alert)
#                    ↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑↑   ← v2 그대로 사용!
print("✅ review_pipeline_v3 (v2 + 알림 라우팅) 완성")

✅ review_pipeline_v3 (v2 + 알림 라우팅) 완성


In [18]:
# 테스트 — 3종 리뷰
reviews = [
    "최악이에요. 박스 열자마자 부품 빠져있고, 환불 요청합니다!!",
    "그냥 그래요. 보통이에요.",
    "정말 만족스러워요! 친구한테도 추천했어요!",
]

for r in reviews:
    out = review_pipeline_v3.invoke({"review": r})
    print(f"{out.action:25s} → #{out.channel}")
    print(f"   📊 {out.summary}")
    print(f"   📝 '{r}'")
    print()

🚨 CS팀 즉시 호출               → #urgent
   📊 [품질] 1/5점 불만 리뷰
   📝 '최악이에요. 박스 열자마자 부품 빠져있고, 환불 요청합니다!!'

📋 일반 로그 저장                → #logs
   📊 [기타] 3/5점
   📝 '그냥 그래요. 보통이에요.'

🎉 마케팅팀에 공유                → #marketing
   📊 긍정도 95점 — 해시태그 활용 가능
   📝 '정말 만족스러워요! 친구한테도 추천했어요!'



**👀 v3 관찰 포인트**

- v2를 그대로 가져다 `| RunnableLambda(route_alert)` 만 추가
- 결과가 **dict가 아닌 `AlertAction` Pydantic 객체** 라서 `.action`, `.channel` 처럼 속성 접근 가능
- LLM 체인 4개의 결과를 종합한 **자동 의사결정** 시스템

다음 v4에서는 여기에 **원본 입력을 함께 보존**해서 완전한 로그 레코드를 만듭니다.


## 6-4. v4 — v3 + 원본 보존 = 완전한 로그 레코드 (RunnablePassthrough)

v3 결과(AlertAction)에 **원본 입력과 분석 객체**를 함께 묶어 DB에 저장 가능한 **완전한 로그 레코드** 를 만듭니다.

이게 오늘의 정점이에요. Step 5에서 배운 `RunnablePassthrough` + 모든 부품 통합.


In [19]:
from datetime import datetime

# 완전한 로그 레코드 스키마 — 정점의 정형 출력
class LogRecord(BaseModel):
    timestamp: str = Field(description="처리 시각 ISO 8601")
    input_review: str = Field(description="고객이 보낸 원본 리뷰")
    rating: int = Field(description="평점")
    sentiment: str = Field(description="감정")
    category: str = Field(description="카테고리")
    sentiment_score: int = Field(description="긍정도 0-100")
    action: str = Field(description="자동 결정된 액션")
    channel: str = Field(description="알림 채널")
    auto_reply: str = Field(description="자동 생성된 답장")

print("✅ LogRecord 스키마 준비 — 모든 결과를 담는 최종 형태")

✅ LogRecord 스키마 준비 — 모든 결과를 담는 최종 형태


In [20]:
# v3 + 원본 + 분석 객체를 함께 묶어 LogRecord로 변환
def to_log_record(combined) -> LogRecord:
    alert = combined["alert"]       # v3 결과 (AlertAction)
    parts = combined["parts"]       # v2 결과 (분석·답장·카테고리·마케팅)
    original = combined["original"] # 원본 입력

    return LogRecord(
        timestamp=datetime.now().isoformat(timespec="seconds"),
        input_review=original["review"],
        rating=parts["analysis"].rating,
        sentiment=parts["analysis"].sentiment,
        category=parts["category"].main_topic,
        sentiment_score=parts["marketing"].sentiment_score,
        action=alert.action,
        channel=alert.channel,
        auto_reply=alert.auto_reply,
    )

In [21]:
# v4: v2·v3·원본을 모두 묶은 뒤 LogRecord 변환
review_pipeline_v4 = RunnableParallel(
    alert=review_pipeline_v3,        # v3 (AlertAction)
    parts=review_pipeline_v2,        # v2 (4개 체인 결과)
    original=RunnablePassthrough(),  # 원본 입력
) | RunnableLambda(to_log_record)

print("✅ review_pipeline_v4 (완전한 로그 레코드 시스템) 완성")

✅ review_pipeline_v4 (완전한 로그 레코드 시스템) 완성


In [22]:
# 테스트 — 최종 결과
import json

record = review_pipeline_v4.invoke({"review": "이 제품 너무 좋아요! 잘 샀어요."})

# Pydantic 객체이므로 .model_dump() 로 dict 변환해서 출력
print(json.dumps(record.model_dump(), ensure_ascii=False, indent=2))

{
  "timestamp": "2026-09-14T14:27:10",
  "input_review": "이 제품 너무 좋아요! 잘 샀어요.",
  "rating": 5,
  "sentiment": "positive",
  "category": "기타",
  "sentiment_score": 95,
  "action": "🎉 마케팅팀에 공유",
  "channel": "marketing",
  "auto_reply": "고객님, 소중한 리뷰 감사합니다! 제품이 마음에 드셨다니 정말 기쁩니다. 앞으로도 더 좋은 제품과 서비스로 보답할 수 있도록 노력하겠습니다. 언제든지 필요하신 점이 있으시면 말씀해 주세요!"
}


**🎉 v4 최종 완성**

오늘의 정점입니다.

| 부품 | 
| --- | --- |
| `chain_analysis` |
| `chain_reply` | 
| `chain_category` |
| `chain_marketing` | 
| `RunnableParallel` | 
| `RunnableLambda` | 
| `RunnablePassthrough` | 
| `PydanticOutputParser` | 
| `Pydantic BaseModel` | 



> 💡 v4 결과는 DB INSERT, JSON 파일 저장, 다음 시스템 전달이 모두 즉시 가능한 **완전한 정형 데이터**입니다. 이게 OutputParser 일관 사용의 진짜 효과예요.


## 6-5. v1~v4 한눈에 비교

```
v1: review_pipeline_v1 = RunnableParallel(분석, 답장, 카테고리)
    → dict {analysis, reply, category}

v2: review_pipeline_v2 = RunnableParallel(분석, 답장, 카테고리, 마케팅)
    → dict {analysis, reply, category, marketing}

v3: review_pipeline_v3 = v2 | RunnableLambda(route_alert)
    → AlertAction 객체

v4: review_pipeline_v4 = RunnableParallel(v3, v2, 원본) | RunnableLambda(to_log_record)
    → LogRecord 객체
```

**핵심**: 각 단계마다 이전 단계의 파이프라인을 **그대로 변수로 가져다 씁니다.** 따로 만드는 게 아니에요. 이게 LCEL의 진짜 매력입니다.


---

# Step 7. 🎯 도전 과제

본인이 관심 있는 도메인을 골라 깊이 시도해보세요. **새로운 체인 + 같은 결합 패턴**으로 만들면 됩니다.

## 과제 A: v4에 체인 하나 더 추가하기 (실무 빈도 ★★★★★)

오늘 만든 `review_pipeline_v4` 에 새로운 체인을 추가해 v5를 만들어보세요.

**아이디어**
- **악성 리뷰 탐지 체인**: 욕설·과한 공격성 여부를 boolean으로
- **카테고리 세분화 체인**: 품질/가격을 더 자세히 분류
- **추천 답변 톤 체인**: 격식/캐주얼/공식 중 어떤 톤이 좋을지

**힌트**: v2에 체인 추가 → v3, v4 자동 반영. 한 줄만 추가하면 됨.

---

## 과제 B: 한 문장 → 다국어 번역 (실무 빈도 ★★★★★)

리뷰 답장을 영어·일본어·중국어로 **동시에** 생성.

```python
multi_lang_pipeline = RunnableParallel(
    ko=chain_reply,
    en=chain_reply_en,
    ja=chain_reply_ja,
)
```

각 체인은 system 메시지의 언어 지시만 다르고 나머지는 똑같아요.

---

## 과제 C: 기사 → 요약 + 카드뉴스 + 해시태그 (실무 빈도 ★★★★☆)

뉴스 기사 하나를 받아 **3가지 형태의 콘텐츠**를 동시 생성. 도메인을 바꿔서 같은 패턴이 통하는지 확인.

- `summary_chain`: 3문장 요약 (StrOutputParser)
- `card_news_chain`: 카드뉴스 5장 (PydanticOutputParser)
- `hashtag_chain`: 해시태그 10개 (PydanticOutputParser)

미디어 회사가 자주 쓰는 패턴.

---

## 과제 D: 본인 도메인 — 다단계 자동화

본인 업무에서 "데이터 1개를 받아 여러 결과를 만들고 싶은 작업"이 있다면 적용해보세요.

- 강의 자료 → 핵심정리 + 발표노트 + 퀴즈
- 회의록 → 액션아이템 + 결정사항 + 후속회의 안건
- 고객 문의 → 답장 초안 + 내부 메모 + 우선순위

**핵심**: 오늘 만든 v1→v4 빌드업 흐름을 본인 도메인에 적용. 각 결과는 Pydantic 스키마로 정형화.

---

> 💡 **하나만 골라서 깊이.** 본인 도메인에 가까울수록 학습 효과 큽니다.
